# Hypothesis Testing, Chi-Square Test, and t-test

Dataset used: Pima Indians Diabetes dataset.

This notebook implements:
- Hypothesis testing framework
- Chi-square test of independence
- One-sample and independent two-sample t-tests

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_style("whitegrid")
df = pd.read_csv("Assignment 3/pima_indians_diabetes.csv")
df.head()

## 1) Data Preparation

We clean medically invalid zero values in selected columns by replacing them with median values.

In [ ]:
cols = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
for c in cols:
    df[c] = df[c].replace(0, np.nan)
    df[c] = df[c].fillna(df[c].median())

df.describe()

In [ ]:
print("Rows:", df.shape[0], "Columns:", df.shape[1])
print("\nOutcome counts:")
print(df["Outcome"].value_counts())

## 2) Chi-Square Test of Independence

Question: Is diabetes outcome associated with glucose category?

- Null hypothesis (H0): Glucose category and outcome are independent.
- Alternate hypothesis (H1): Glucose category and outcome are associated.
- Significance level: 0.05

In [ ]:
df["GlucoseCategory"] = pd.cut(
    df["Glucose"],
    bins=[0, 99, 125, 300],
    labels=["Normal", "Prediabetes", "Diabetes"]
)

contingency = pd.crosstab(df["GlucoseCategory"], df["Outcome"])
chi2, p_value, dof, expected = stats.chi2_contingency(contingency)

print("Contingency Table:")
print(contingency)
print("\nChi-square statistic:", round(chi2, 4))
print("p-value:", round(p_value, 6))
print("Degrees of freedom:", dof)

if p_value < 0.05:
    print("Result: Reject H0 -> Significant association")
else:
    print("Result: Fail to reject H0 -> No significant association")

pd.DataFrame(expected, index=contingency.index, columns=contingency.columns)

## 3) One-Sample t-test

Question: Is average glucose significantly different from 110?

- Null hypothesis (H0): Mean glucose = 110
- Alternate hypothesis (H1): Mean glucose != 110
- Significance level: 0.05

In [ ]:
t_stat_1, p_value_1 = stats.ttest_1samp(df["Glucose"], popmean=110)
print("Sample mean glucose:", round(df["Glucose"].mean(), 3))
print("t-statistic:", round(t_stat_1, 4))
print("p-value:", round(p_value_1, 6))

if p_value_1 < 0.05:
    print("Result: Reject H0 -> Mean glucose is significantly different from 110")
else:
    print("Result: Fail to reject H0 -> No significant difference from 110")

## 4) Independent Two-Sample t-test

Question: Is mean BMI different between diabetic and non-diabetic groups?

- Null hypothesis (H0): Mean BMI (Outcome=1) = Mean BMI (Outcome=0)
- Alternate hypothesis (H1): Means are different
- Significance level: 0.05

In [ ]:
bmi_diabetic = df[df["Outcome"] == 1]["BMI"]
bmi_non_diabetic = df[df["Outcome"] == 0]["BMI"]

t_stat_2, p_value_2 = stats.ttest_ind(bmi_diabetic, bmi_non_diabetic, equal_var=False)
print("Mean BMI (Outcome=1):", round(bmi_diabetic.mean(), 3))
print("Mean BMI (Outcome=0):", round(bmi_non_diabetic.mean(), 3))
print("t-statistic:", round(t_stat_2, 4))
print("p-value:", round(p_value_2, 6))

if p_value_2 < 0.05:
    print("Result: Reject H0 -> Mean BMI differs between groups")
else:
    print("Result: Fail to reject H0 -> No significant difference in mean BMI")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.countplot(data=df, x="GlucoseCategory", hue="Outcome", ax=axes[0])
axes[0].set_title("Outcome by Glucose Category")

sns.boxplot(data=df, x="Outcome", y="BMI", ax=axes[1])
axes[1].set_title("BMI by Outcome")

plt.tight_layout()
plt.show()

## 5) Final Conclusion

- The Chi-square test checks association between categorical variables.
- The one-sample t-test checks if a sample mean differs from a reference value.
- The independent t-test compares means of two independent groups.
- Decisions are made using p-value and significance level (0.05).